# Byonoy Absorbance 96

The Absorbance 96 Automate (A96A) is a USB-HID plate reader from Byonoy that measures absorbance across a 96-well plate in a single flash. It supports:

- [Absorbance](../../capabilities/absorbance) (single-wavelength, full-plate)

The hardware consists of three physical parts: a **base unit** (holds the plate), an **illumination unit** (light source, sits on top during measurement), and an optional **SBS adapter** for standard footprint integration. PLR models all three as resources so a robotic arm can move the illumination unit on and off the base.

| Model | PLR Name | Factory function |
|---|---|---|
| Absorbance 96 Automate (full setup) | `ByonoyAbsorbance96` | `byonoy_a96a` |
| Detection unit only | `ByonoyAbsorbance96` | `byonoy_a96a_detection_unit` |
| Illumination unit | `Resource` | `byonoy_a96a_illumination_unit` |
| Parking base (no backend) | `ByonoyAbsorbanceBaseUnit` | `byonoy_a96a_parking_unit` |
| SBS adapter | `ResourceHolder` | `byonoy_sbs_adapter` |

- **Communication**: USB HID (VID `0x16D0` / PID `0x1199`)
- **Communication level**: Firmware

## Setup

Use `byonoy_a96a` to create the full setup (detection unit + illumination unit). The detection unit is both a `Resource` (base with plate holder) and a `Device` (drives the backend over USB HID).

In [ ]:
from pylabrobot.byonoy import byonoy_a96a

reader, illumination_unit = byonoy_a96a(name="a96a")
await reader.setup()

During `setup()`, the backend opens the USB HID connection and runs an initialization measurement to calibrate the sensor.

## Absorbance

The absorbance capability is exposed as `reader.absorbance`. For the full API, see [Absorbance](../../capabilities/absorbance).

Before reading, assign a plate to the base unit's plate holder and make sure the illumination unit is removed from the base (the interlock will raise an error otherwise).

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

# Remove the illumination unit from the base so the plate can be loaded
reader.illumination_unit_holder.unassign_child_resource(illumination_unit)

plate = Cor_96_wellplate_360ul_Fb(name="plate")
reader.plate_holder.assign_child_resource(plate)

Read absorbance at a single wavelength. The wavelength must be one of the device's available wavelengths (queried automatically during `setup()`).

In [ ]:
results = await reader.absorbance.read_absorbance(plate, wavelength=450)

You can check which wavelengths are available on your unit:

In [ ]:
print(reader.driver.available_wavelengths)

## Resource layout

The Absorbance 96 has an interlock: you cannot assign a plate to the base while the illumination unit is on top. In an automated workcell, use a robotic arm to move the illumination unit to a parking base before loading the plate.

```
ByonoyAbsorbance96 (base + device)
 +-- plate_holder              (assign plates here)
 +-- illumination_unit_holder  (illumination unit sits here during measurement)
```

A separate `byonoy_a96a_parking_unit` can be used as a second base (no backend) where the illumination unit rests while plates are swapped.

## Teardown

In [ ]:
await reader.stop()